In [2]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import car_enrichment as ce

In [3]:
with open("car_price_model.pkl", "rb") as f:
    xgb_loaded = pickle.load(f)

In [4]:
"""
car_price_predictor.py

Turns a simple, human-friendly car description (Year, Make, Model, mileage,
transmission, plus a few optional condition flags) into a price prediction
from your trained model.

Handles all the lookups your pipeline needs behind the scenes:
    - Age = CURRENT_YEAR - Year   (CURRENT_YEAR = 2027, confirmed)
    - Cylinders, Body_Type  -> looked up from EngineType_enriched.xlsx
                                (mode per Make/Model, since trims vary)
    - std_tier              -> looked up from Model_Flag.csv
                                ("0std" and any unseen Make/Model both map
                                to "Normal" - your training data has no
                                std_tier_0std column, so this mapping is
                                required, not optional)

IMPORTANT: this module validates every generated category against your
ACTUAL training_columns list. If a lookup produces a value with no
matching dummy column in training_columns (e.g. a Cylinders value your
model never saw), it prints a clear warning instead of silently zeroing
out that feature - this is the exact bug class that caused prior wrong
predictions (std_tier_0std, missing transmission input).

Usage:
    import pickle
    import pandas as pd
    from car_price_predictor import load_lookups, predict_car_price

    with open("car_price_model.pkl", "rb") as f:
        model = pickle.load(f)

    training_columns = pd.read_csv("training_columns.csv")["column"].tolist()
    lookups = load_lookups("EngineType_enriched.xlsx", "Model_Flag.csv")

    price = predict_car_price(
        year=2024, make="Chevrolet", model_name="Malibu", mileage=71962,
        transmission="Automatic",
        model=model, training_columns=training_columns, lookups=lookups,
        has_carfax=1, no_accidents=1
    )
    print(price)
"""

import pandas as pd
import numpy as np

CURRENT_YEAR = 2027  # confirmed correct


def load_lookups(engine_file="EngineType_enriched.xlsx", model_flag_file="Model_Flag.csv"):
    """
    Loads and prepares the two reference tables into lookup structures.
    """
    engine_df = pd.read_excel(engine_file)
    engine_df["Make"] = engine_df["Make"].astype(str)
    engine_df["Model"] = engine_df["Model"].astype(str)

    def safe_mode(series):
        m = series.mode()
        return m.iloc[0] if not m.empty else None

    engine_lookup = engine_df.groupby(["Make", "Model"]).agg(
        Cylinders=("Cylinders", safe_mode),
        Body_Type=("Body Type", safe_mode)
    )

    model_flags = pd.read_csv(model_flag_file)
    model_flags["Make"] = model_flags["Make"].astype(str)
    model_flags["Model"] = model_flags["Model"].astype(str)

    # IMPORTANT: your trained model has NO std_tier_0std column - only
    # 1std / 2std / 3std / Normal. "0std" (below-average models) must be
    # folded into "Normal" to match what the model actually learned.
    model_flags["std_tier"] = model_flags["std_tier"].replace("0std", "Normal")

    std_tier_lookup = {
        (row["Make"], row["Model"]): row["std_tier"]
        for _, row in model_flags.iterrows()
    }

    return {
        "engine_lookup": engine_lookup,
        "std_tier_lookup": std_tier_lookup,
    }


def _validate_category(value, prefix, training_columns, field_label):
    """
    Checks that `prefix + value` (e.g. "Cylinders_6") exists among
    training_columns. Warns loudly if not, since that means this
    category will silently zero out instead of being represented.
    """
    expected_col = f"{prefix}_{value}"
    valid_values = sorted({
        col[len(prefix) + 1:] for col in training_columns if col.startswith(prefix + "_")
    })

    if expected_col not in training_columns:
        print(f"WARNING: {field_label} = '{value}' has no matching training column "
              f"('{expected_col}' not found). This feature will be zeroed out for "
              f"this prediction, which may cause an inaccurate result. "
              f"Valid trained categories for {field_label}: {valid_values}")
    return value


def check_car_lookup(make, model_name, lookups):
    """
    Quick diagnostic: shows exactly what Cylinders, Body_Type, and std_tier
    were resolved for a given Make/Model, without needing mileage, year,
    training_columns, or a model. Useful for verifying your merges/lookups
    are correct before trusting a full prediction.

    Prints the result and also returns it as a dict.
    """
    make = str(make)
    model_name = str(model_name)

    try:
        row = lookups["engine_lookup"].loc[(make, model_name)]
        cylinders = row["Cylinders"]
        body_type = row["Body_Type"]
        found_engine_data = True
    except KeyError:
        cylinders = "Unknown"
        body_type = "Unknown"
        found_engine_data = False

    raw_std_tier = None
    for (m, mdl), tier in lookups["std_tier_lookup"].items():
        if m == make and mdl == model_name:
            raw_std_tier = tier
            break
    std_tier = raw_std_tier if raw_std_tier is not None else "Normal"

    print(f"Make: {make}")
    print(f"Model: {model_name}")
    print(f"Cylinders: {cylinders}" + ("" if found_engine_data else "  (no engine data found - defaulted)"))
    print(f"Body_Type: {body_type}" + ("" if found_engine_data else "  (no engine data found - defaulted)"))
    print(f"std_tier: {std_tier}")

    return {
        "Make": make,
        "Model": model_name,
        "Cylinders": cylinders,
        "Body_Type": body_type,
        "std_tier": std_tier,
        "found_engine_data": found_engine_data,
    }


def build_input_row(year, make, model_name, mileage, lookups, training_columns,
                      transmission="Automatic",
                      no_accidents=0, has_carfax=0, one_owner=0,
                      service_records=0, certified=0, is_hybrid=0):
    """
    Assembles a single-row DataFrame matching the model's real trained
    feature set, validating every looked-up category against
    training_columns as it goes.
    """
    make = str(make)
    model_name = str(model_name)

    age = CURRENT_YEAR - year

    # --- Engine / Body Type lookup ---
    try:
        row = lookups["engine_lookup"].loc[(make, model_name)]
        cylinders = row["Cylinders"]
        body_type = row["Body_Type"]
    except KeyError:
        print(f"Warning: no engine/body-type data found for {make} {model_name} — "
              f"using 'Unknown' for both. Predictions may be less reliable.")
        cylinders = "Unknown"
        body_type = "Unknown"

    # --- std_tier lookup (0std and unseen models both fold into "Normal") ---
    std_tier = lookups["std_tier_lookup"].get((make, model_name), "Normal")

    # --- Validate every categorical value against the model's real trained columns ---
    _validate_category(cylinders, "Cylinders", training_columns, "Cylinders")
    _validate_category(body_type, "Body Type", training_columns, "Body_Type")
    _validate_category(std_tier, "std_tier", training_columns, "std_tier")
    _validate_category(make, "Make", training_columns, "Make")

    transmission_manual = 1 if str(transmission).strip().lower() == "manual" else 0

    input_row = pd.DataFrame([{
        "Make": make,
        "mileage": mileage,
        "Age": age,
        "no_accidents": no_accidents,
        "has_carfax": has_carfax,
        "one_owner": one_owner,
        "service_records": service_records,
        "certified": certified,
        "Is_Hybrid": is_hybrid,
        "transmission_Manual": transmission_manual,
        "std_tier": std_tier,
        "Body Type": body_type,
        "Cylinders": cylinders,
    }])

    return input_row


def predict_car_price(year, make, model_name, mileage, model, training_columns, lookups,
                        transmission="Automatic",
                        no_accidents=0, has_carfax=0, one_owner=0,
                        service_records=0, certified=0, is_hybrid=0,
                        log_transformed=True):
    """
    Full pipeline: raw car description -> predicted price.
    See module docstring for parameter details.
    """
    input_row = build_input_row(
        year, make, model_name, mileage, lookups, training_columns,
        transmission, no_accidents, has_carfax, one_owner,
        service_records, certified, is_hybrid
    )

    # Dummy-encode the categorical fields (Make, std_tier, Body Type, Cylinders),
    # keep the rest (already numeric/binary) as-is, then align exactly to
    # training_columns - any column not generated here defaults to 0.
    input_encoded = pd.get_dummies(input_row, columns=["Make", "std_tier", "Body Type", "Cylinders"])
    input_encoded = input_encoded.reindex(columns=training_columns, fill_value=0)

    pred = model.predict(input_encoded)

    if log_transformed:
        return float(np.expm1(pred)[0])
    return float(pred[0])

In [32]:
#What Do I want to give it as an Input?? to my function and What parts the program itself should go and find?
with open("car_price_model.pkl", "rb") as f:
    model = pickle.load(f)

# Save this once, right after training: pd.Series(X_train.columns).to_csv("training_columns.csv", index=False)
training_columns = pd.read_csv("training_columns.csv")["0"].tolist()

lookups = load_lookups("EngineType_enriched.xlsx", "Model_Flag.csv")

price = predict_car_price(
    year=2024, make="Chevrolet", model_name="Trax", mileage=48000,
    model=model, training_columns=training_columns, lookups=lookups,
    has_carfax=0, no_accidents=0,certified=0,is_hybrid=0,one_owner=0)
print(price)

26512.505859375


In [6]:
lookups = load_lookups("EngineType_enriched.xlsx", "Model_Flag.csv")

check_car_lookup("Mercedes-Benz", "S-Class", lookups)

Make: Mercedes-Benz
Model: S-Class
Cylinders: 6
Body_Type: Sedan
std_tier: Normal


{'Make': 'Mercedes-Benz',
 'Model': 'S-Class',
 'Cylinders': '6',
 'Body_Type': 'Sedan',
 'std_tier': 'Normal',
 'found_engine_data': True}

In [27]:
training_columns[20:50]

['Make_Ford',
 'Make_GMC',
 'Make_Genesis',
 'Make_Honda',
 'Make_Hyundai',
 'Make_Infiniti',
 'Make_Jeep',
 'Make_Kia',
 'Make_Lexus',
 'Make_Lincoln',
 'Make_MINI',
 'Make_Mazda',
 'Make_Mercedes-Benz',
 'Make_Mitsubishi',
 'Make_Nissan',
 'Make_Porsche',
 'Make_RAM',
 'Make_Subaru',
 'Make_Toyota',
 'Make_Volkswagen',
 'Make_Volvo',
 'Body Type_Coupe',
 'Body Type_Hatchback',
 'Body Type_Minivan',
 'Body Type_Pickup Truck',
 'Body Type_SUV',
 'Body Type_Sedan',
 'Body Type_Van',
 'Body Type_Wagon',
 'Cylinders_3']